# Notebook 01: Data Audit
##### **PURPOSE:**
###### Profile all 11 raw CSV files BEFORE any cleaning happens.
###### The goal is to answer: "What exactly is wrong with this data?

##### **WHAT THIS NOTEBOOK DOES:**
###### - Counts rows and columns in each tables
###### - Measures what percentage of each column is null
###### - Detects common data quality problems automatically
###### (ALL CAPS email, mixed currency formats, mixed data formats,
###### negative values in numeric columns, EU decimal format)
###### - Checks whether JOIN keys are consistent across tables
###### (email casing, SKU casing, orphan foreign keys)
###### - Saves two output fields

##### **OUTPUT FIELDS** (saved to GGDrive)


## Mount Google Drive
###### Gives Colab access to the Drive files

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configure Paths
###### RAW_DIR   = the folder where 11 original CSV files are stored
###### BASE_DIR  = the praoject root folder on the Drive the audit/ subfolder will be created automatically inside it

In [ ]:
import os

RAW_DIR   = '/content/drive/MyDrive/AmazinChoices/datasets'
BASE_DIR  = '/content/drive/MyDrive/AmazinChoices'

AUDIT_DIR = os.path.join(BASE_DIR, 'audit')
os.makedirs(AUDIT_DIR, exist_ok=True)

ALL_TABLES = [
    'orders',
    'order_items',
    'customer_profiles',
    'customer_interactions',
    'marketing_spend',
    'refunds',
    'cost_history',
    'product_master',
    'promotions_log',
    'purchase_orders',
    'shipping_rates',
]

#Verify that every expected file actually exists before we start
print("Checking files in RAW_DIR: " + RAW_DIR)
print("")

all_found = True
for table_name in ALL_TABLES:
    file_path = os.path.join(RAW_DIR, table_name + '.csv')
    file_exists = os.path.exists(file_path)
    if file_exists:
        file_size_mb = round(os.path.getsize(file_path) / 1024 / 1024, 1)
        print("  FOUND   " + table_name + ".csv  (" + str(file_size_mb) + " MB)")
    else:
        print("  MISSING " + table_name + ".csv")
        all_found = False

print("")
if all_found:
    print("All 11 files found. Ready to run the audit.")
else:
    print("Some files are missing. Fix RAW_DIR before continuing.")

Checking files in RAW_DIR: /content/drive/MyDrive/AmazinChoices/datasets

  FOUND   orders.csv  (15.0 MB)
  FOUND   order_items.csv  (12.8 MB)
  FOUND   customer_profiles.csv  (5.4 MB)
  FOUND   customer_interactions.csv  (11.0 MB)
  FOUND   marketing_spend.csv  (5.5 MB)
  FOUND   refunds.csv  (7.7 MB)
  FOUND   cost_history.csv  (4.5 MB)
  FOUND   product_master.csv  (5.1 MB)
  FOUND   promotions_log.csv  (5.5 MB)
  FOUND   purchase_orders.csv  (7.7 MB)
  FOUND   shipping_rates.csv  (3.4 MB)

All 11 files found. Ready to run the audit.


## Import libraries
###### pandas  = the main data analysis library, used to read and inspect CSV files
###### json    = used to save the audit report in a structured format
###### os      = used to check file paths and create folders
###### warnings = we suppress harmless warnings to keep the output clean

In [ ]:
import pandas as pd
import json
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

## Define the audit functions
###### detect_issues(df)
###### Scans every column in a dataframe for known data quality patterns

###### audit_one_table(table_name, raw_dir)
###### loads one CSV file and runs a full profile on it
###### calls "detect_issues()" internally

In [ ]:
def detect_issues(df):
  """Problems this function checks for:
      1. ALL CAPS email addresses
         These will silently fail when you JOIN to customer_profiles
         because 'ELLA@GMAIL.COM' does not equal 'ella@gmail.com'.

      2. Currency symbols in numeric columns
         Values like '$8.84' or 'USD 15.00' cannot be converted to float
         without removing the symbol first.

      3. EU decimal format
         '7,73' looks like a number with a thousands comma but it is actually
         7.73 using European notation where comma means decimal point.
         This must be handled separately from '1,347' which is 1347 US style.

      4. Mixed date formats in the same column
         A single column may contain '2025-01-08' and '10/23/25' and '3-Nov-25'
         at the same time. pd.to_datetime() alone cannot parse all of these.

      5. Negative values in columns that should always be positive
         For example, a negative unit_cost or negative shipping_fee is almost
         certainly a data entry error and needs to be corrected.
    """
  issues_found = []

  for col in df.columns:
    non_null_values = df[col].dropna().astype(str)

    if len(non_null_values) == 0:
      continue

    #Check 1: ALL CAPS email addresses
    if 'email' in col.lower():
      pct_all_caps = non_null_values.str.isupper().mean()
      if pct_all_caps >= 0.3:
        issues_found.append(
            col + ": " + str(round(pct_all_caps * 100)) + "% of values are ALL CAPS. "
        )

     # Check 2: Currency symbols
        has_dollar = non_null_values.str.startswith('$').any()
        has_usd_prefix = non_null_values.str.startswith('USD').any()
        has_eur_prefix = non_null_values.str.startswith('EUR').any()
        if has_dollar or has_usd_prefix or has_eur_prefix:
            issues_found.append(
                col + ": contains currency symbols ($ or USD or EUR). "
                "Must strip symbols before converting to float."
            )

      # Check 3: EU decimal format  (e.g. '7,73' meaning 7.73)
        eu_decimal_pattern = non_null_values.str.match(r'^\d+,\d{2}$')
        if eu_decimal_pattern.any():
            issues_found.append(
                col + ": EU decimal format detected (example: '7,73' meaning 7.73). "
                "Comma here is a decimal point, not a thousands separator. "
                "This needs special handling separate from US thousands commas."
            )

      # Check 4: Mixed date formats
        date_formats_found = []
        if non_null_values.str.match(r'\d{1,2}/\d{1,2}/\d{2,4}').any():
            date_formats_found.append('M/D/YY')
        if non_null_values.str.match(r'\d{1,2}-[A-Za-z]{3}-\d{2,4}').any():
            date_formats_found.append('D-Mon-YY')
        if non_null_values.str.match(r'\d{4}-\d{2}-\d{2}T').any():
            date_formats_found.append('ISO-8601')
        if non_null_values.str.match(r'\d{2}-\d{2}-\d{4}').any():
            date_formats_found.append('D-M-YYYY')
        if len(date_formats_found) > 1:
            issues_found.append(
                col + ": mixed date formats in the same column: " + ', '.join(date_formats_found) + ". "
                "pd.to_datetime() alone will fail on some rows. "
                "Use parse_date() from 02_data_cleaning.ipynb instead."
            )

      # Check 5: Negative values where they should not exist
        numeric_attempt = pd.to_numeric(
            non_null_values.str.replace(r'[^\d.\-]', '', regex=True),
            errors='coerce'
        )
        if numeric_attempt.notna().any():
            pct_negative = (numeric_attempt < 0).mean()
            if pct_negative > 0:
                issues_found.append(
                    col + ": " + str(round(pct_negative * 100, 1)) + "% of values are negative. "
                    "If this column should always be positive, use .clip(lower=0) or .abs() when cleaning."
                )

    return issues_found

In [ ]:
def audit_one_table(table_name, raw_dir):
    """
    Load one raw CSV and produce a full profile.
    Return a dictionary with all findings.

    The dictionary contains:
      table       : table name
      rows        : total row count
      cols        : total column count
      file_mb     : file size in megabytes
      columns     : list of column names
      null_pct    : dict mapping column name to null percentage
      dtypes      : dict mapping column name to pandas dtype
      n_unique    : dict mapping column name to unique value count
      samples     : dict mapping column name to up to 5 sample values
      issues      : list of plain English problem descriptions
    """
    file_path = os.path.join(raw_dir, table_name + '.csv')
    df = pd.read_csv(file_path, low_memory=False)

    null_pct = df.isnull().mean().mul(100).round(2).to_dict()
    dtypes   = df.dtypes.astype(str).to_dict()
    n_unique = df.nunique().to_dict()
    file_mb  = round(os.path.getsize(file_path) / 1024 / 1024, 2)

    samples = {}
    for col in df.columns:
        unique_values = df[col].dropna().unique()
        samples[col] = [str(v) for v in unique_values[:5]]

    issues = detect_issues(df)

    return {
        'table':    table_name,
        'rows':     len(df),
        'cols':     df.shape[1],
        'file_mb':  file_mb,
        'columns':  list(df.columns),
        'null_pct': null_pct,
        'dtypes':   dtypes,
        'n_unique': n_unique,
        'samples':  samples,
        'issues':   issues,
    }


print("Audit functions defined.")

Audit functions defined.


## Run the audit on all 11 tables
###### This cell loops through all 11 tables, calls audit_one_table() on each,
###### and prints a summary to the screen.

###### For each table you will see:
######   - Row count, column count, file size
######   - Null percentage for every column that has nulls
######   - Issues detected automatically by detect_issues()
######   - Sample values from the first few columns


In [ ]:
all_audit_results = {}

for table_name in ALL_TABLES:

    result = audit_one_table(table_name, RAW_DIR)
    all_audit_results[table_name] = result
    # Print table header
    print("")
    print("-" * 60)
    print(table_name.upper())
    print(str(result['rows']) + " rows   " + str(result['cols']) + " columns   " + str(result['file_mb']) + " MB")
    print("-" * 60)

    # Print null percentages for columns that have at least one null
    columns_with_nulls = {col: pct for col, pct in result['null_pct'].items() if pct > 0}
    if columns_with_nulls:
        print("Null percentages:")
        for col, pct in sorted(columns_with_nulls.items(), key=lambda x: -x[1]):
            high_flag = "  <-- HIGH" if pct > 20 else ""
            print("  " + col.ljust(32) + str(pct) + "%" + high_flag)
    else:
        print("Null percentages: none")

    # Print issues detected automatically
    if result['issues']:
        print("Issues detected:")
        for issue in result['issues']:
            print("  ISSUE: " + issue)
    else:
        print("Issues detected: none")

    # Print sample values from the first 4 columns
    print("Sample values (first 4 columns):")
    for col in list(result['samples'].keys())[:4]:
        print("  " + col.ljust(30) + str(result['samples'][col]))

NOTEBOOK 01: DATA AUDIT
Started: 2026-03-03 22:58

------------------------------------------------------------
ORDERS
110000 rows   11 columns   14.98 MB
------------------------------------------------------------
Null percentages:
  shipping_address_note           71.11%  <-- HIGH
  promo_code_raw                  21.31%  <-- HIGH
  shipping_fee_raw                11.94%
  tax_raw                         11.83%
Issues detected: none
Sample values (first 4 columns):
  order_id                      ['ord_e4ba85840c28377531b65a40_1', 'ord_eebf98ba0ca6e633e6fdc5d0_2', 'ord_000864f6ef9fa6c8cc1c4fa3_3', 'ord_10ffd01968b158c33966f153_4', 'ord_7ac998b2cd65e87c1920fe83_5']
  platform                      ['Website', 'Walmart', 'Instagram', 'Local', 'Amazon']
  platform_order_id             ['web_6c822b4086a8f8c21360d0f93fb0', '531-1439384-3851888', 'web_68176887e59fa9fb41672b1fe0c9', 'IN_340c1facfb51a2c930', 'IN_761dc06e76627e2f4e']
  order_timestamp               ['2025-01-08T03:21:49-08:00

## Check cross-table JOIN key consistency


In [ ]:
# Load the four tables we need for these checks
orders_raw   = pd.read_csv(os.path.join(RAW_DIR, 'orders.csv'),           low_memory=False)
items_raw    = pd.read_csv(os.path.join(RAW_DIR, 'order_items.csv'),      low_memory=False)
customers_raw = pd.read_csv(os.path.join(RAW_DIR, 'customer_profiles.csv'), low_memory=False)
refunds_raw  = pd.read_csv(os.path.join(RAW_DIR, 'refunds.csv'),           low_memory=False)
products_raw = pd.read_csv(os.path.join(RAW_DIR, 'product_master.csv'),    low_memory=False)

cross_issues = []

In [ ]:
# Check 1: Email casing between orders and customer_profiles
orders_emails    = orders_raw['customer_email'].dropna()
customers_emails = customers_raw['customer_email'].dropna()

direct_match_pct = orders_emails.isin(customers_emails).mean() * 100
lower_match_pct  = orders_emails.str.lower().isin(customers_emails.str.lower()).mean() * 100

print("")
print("CHECK 1: orders.customer_email vs customer_profiles.customer_email")
print("  Match rate (case-sensitive): " + str(round(direct_match_pct, 1)) + "%")
print("  Match rate (after lowercase): " + str(round(lower_match_pct, 1)) + "%")

if direct_match_pct < lower_match_pct - 5:
    problem = (
        "Email casing mismatch. Only " + str(round(direct_match_pct, 0)) + "% match directly, "
        "but " + str(round(lower_match_pct, 0)) + "% match after converting to lowercase. "
        "Fix: normalize both tables with .str.lower().str.strip() before joining."
    )
    cross_issues.append(problem)
    print("  ISSUE: " + problem)
else:
    print("  OK: email casing is consistent across both tables")


CHECK 1: orders.customer_email vs customer_profiles.customer_email
  Match rate (case-sensitive): 100.0%
  Match rate (after lowercase): 100.0%
  OK: email casing is consistent across both tables


In [ ]:
# Check 2: SKU casing between order_items and product_master
items_sku    = items_raw['sku_raw'].dropna().str.strip()
products_sku = products_raw['sku'].dropna().str.strip()

sku_direct_pct = items_sku.isin(products_sku).mean() * 100
sku_upper_pct  = items_sku.str.upper().isin(products_sku.str.upper()).mean() * 100

print("")
print("CHECK 2: order_items.sku vs product_master.sku")
print("  Match rate (case-sensitive): " + str(round(sku_direct_pct, 1)) + "%")
print("  Match rate (after uppercase): " + str(round(sku_upper_pct, 1)) + "%")

if sku_direct_pct < sku_upper_pct - 5:
    problem = (
        "SKU casing mismatch. "
        "Fix: uppercase both tables with .str.upper().str.strip() before joining."
    )
    cross_issues.append(problem)
    print("  ISSUE: " + problem)
else:
    print("  OK: SKU casing is consistent across both tables")


CHECK 2: order_items.sku vs product_master.sku
  Match rate (case-sensitive): 100.0%
  Match rate (after uppercase): 100.0%
  OK: SKU casing is consistent across both tables


In [ ]:
# Check 3: Orphan order_ids in order_items
orders_ids    = orders_raw['order_id'].dropna()
items_ids     = items_raw['order_id'].dropna()
orphan_items_pct = (~items_ids.isin(orders_ids)).mean() * 100

print("")
print("CHECK 3: order_items.order_id -> orders.order_id")
print("  Orphan rate (items with no matching order): " + str(round(orphan_items_pct, 1)) + "%")
if orphan_items_pct > 5:
    problem = str(round(orphan_items_pct, 0)) + "% of order_item rows have no matching order. Investigate before analysis."
    cross_issues.append(problem)
    print("  ISSUE: " + problem)
else:
    print("  OK: all or almost all order_item rows have a matching order")


CHECK 3: order_items.order_id -> orders.order_id
  Orphan rate (items with no matching order): 0.0%
  OK: all or almost all order_item rows have a matching order


In [ ]:
# Check 4: Orphan order_ids in refunds
refunds_ids     = refunds_raw['order_id'].dropna()
orphan_refs_pct = (~refunds_ids.isin(orders_ids)).mean() * 100

print("")
print("CHECK 4: refunds.order_id -> orders.order_id")
print("  Orphan rate (refunds with no matching order): " + str(round(orphan_refs_pct, 1)) + "%")
if orphan_refs_pct > 5:
    problem = str(round(orphan_refs_pct, 0)) + "% of refund rows have no matching order. Investigate before analysis."
    cross_issues.append(problem)
    print("  ISSUE: " + problem)
else:
    print("  OK: all or almost all refund rows have a matching order")

print("")
print("Cross-table issues found: " + str(len(cross_issues)))


CHECK 4: refunds.order_id -> orders.order_id
  Orphan rate (refunds with no matching order): 0.0%
  OK: all or almost all refund rows have a matching order

Cross-table issues found: 0


## Save audit results

In [ ]:
all_audit_results['_cross_table_issues'] = cross_issues
all_audit_results['_meta'] = {
    'run_at':       str(datetime.now()),
    'raw_dir':      RAW_DIR,
    'total_tables': len(ALL_TABLES),
    'total_rows':   sum(all_audit_results[name]['rows'] for name in ALL_TABLES),
}

# Save JSON report
json_path = os.path.join(AUDIT_DIR, '01_audit_report.json')
with open(json_path, 'w') as f:
    json.dump(all_audit_results, f, indent=2, default=str)

# Save plain text summary
txt_path = os.path.join(AUDIT_DIR, '01_audit_summary.txt')
with open(txt_path, 'w') as f:
    f.write("IGNIRA DATA AUDIT REPORT\n")
    f.write("Run at: " + all_audit_results['_meta']['run_at'] + "\n")
    f.write("Raw data folder: " + RAW_DIR + "\n")
    f.write("=" * 60 + "\n\n")
    f.write("SUMMARY\n")
    f.write("Total tables: " + str(len(ALL_TABLES)) + "\n")
    f.write("Total rows:   " + str(all_audit_results['_meta']['total_rows']) + "\n\n")
    f.write("TABLE DETAILS\n")
    f.write("-" * 60 + "\n")

    for name in ALL_TABLES:
        r = all_audit_results[name]
        f.write("\n" + name.upper() + "\n")
        f.write(str(r['rows']) + " rows  " + str(r['cols']) + " columns  " + str(r['file_mb']) + " MB\n")

        columns_with_nulls = {c: v for c, v in r['null_pct'].items() if v > 0}
        if columns_with_nulls:
            f.write("Null percentages:\n")
            for col, pct in sorted(columns_with_nulls.items(), key=lambda x: -x[1]):
                f.write("  " + col + ": " + str(pct) + "%\n")

        if r['issues']:
            f.write("Issues:\n")
            for issue in r['issues']:
                f.write("  ISSUE: " + issue + "\n")

    if cross_issues:
        f.write("\nCROSS-TABLE ISSUES\n")
        f.write("-" * 60 + "\n")
        for issue in cross_issues:
            f.write("  ISSUE: " + issue + "\n")

total_rows = all_audit_results['_meta']['total_rows']
total_issues = sum(len(all_audit_results[name]['issues']) for name in ALL_TABLES) + len(cross_issues)

print("")
print("=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)
print("Tables profiled:  " + str(len(ALL_TABLES)))
print("Total rows:       " + str(total_rows))
print("Total issues:     " + str(total_issues))
print("")
print("Files saved:")
print("  " + json_path)
print("  " + txt_path)


AUDIT COMPLETE
Tables profiled:  11
Total rows:       1045000
Total issues:     0

Files saved:
  /content/drive/MyDrive/AmazinChoices/audit/01_audit_report.json
  /content/drive/MyDrive/AmazinChoices/audit/01_audit_summary.txt
